In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime

# 16 CPUS
# 128 GB Memory

spark = SparkSession.builder \
    .appName("PushshiftRedditEDA") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.instances", 15) \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "64") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Matplotlib created a temporary cache directory at /scratch/jkeeton/job_48220095/matplotlib-jyslefj0 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Spark version: 3.5.0
Spark UI: http://exp-2-37.expanse.sdsc.edu:4040


In [2]:
import os
import glob
from pyspark.sql import functions as F

DATA_DIR = "/expanse/lustre/scratch/jkeeton/temp_project/pushshift-reddit/data/"
files = sorted(glob.glob(DATA_DIR + "*.parquet"))

COLS = ["author", "created_utc", "id", "num_comments", "score", 
        "selftext", "subreddit", "subreddit_id", "title"]

def read_one(path):
    return spark.read.parquet(path) \
        .select(*[F.col(c) for c in COLS if c != "created_utc"],
                F.col("created_utc").cast("long").alias("created_utc"))

# Read and union all files
dfs = [read_one(f) for f in files]
df = dfs[0]
for d in dfs[1:]:
    df = df.unionByName(d)

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")


Files loaded: 218
Partitions: 3339


In [3]:
df.printSchema()

root
 |-- author: string (nullable = true)
 |-- id: string (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- created_utc: long (nullable = true)



In [4]:
row_count = df.count()
print(f"Total observations: {row_count:,}")

Total observations: 549,662,955


In [5]:
df.select("score", "num_comments").summary("count", "mean", "stddev", "min", "max").show()

+-------+-----------------+-----------------+
|summary|            score|     num_comments|
+-------+-----------------+-----------------+
|  count|        549662934|        549662955|
|   mean| 44.8390808193008|8.358100421739355|
| stddev|707.3788392973761|93.11190659178932|
|    min|                0|             -117|
|    max|           270469|           517003|
+-------+-----------------+-----------------+



In [6]:
df.select(
    F.count("created_utc").alias("non_null"),
    F.to_timestamp(F.min("created_utc")).alias("min_utc"),
    F.to_timestamp(F.max("created_utc")).alias("max_utc"),
).show()

+---------+-------------------+-------------------+
| non_null|            min_utc|            max_utc|
+---------+-------------------+-------------------+
|549662955|2012-01-01 00:00:01|2018-12-31 23:59:59|
+---------+-------------------+-------------------+



In [8]:
for c in COLS:
    print(c)
    df.select(F.count(c)).show()

author
+-------------+
|count(author)|
+-------------+
|    549662955|
+-------------+

created_utc
+------------------+
|count(created_utc)|
+------------------+
|         549662955|
+------------------+

id
+---------+
|count(id)|
+---------+
|549662955|
+---------+

num_comments
+-------------------+
|count(num_comments)|
+-------------------+
|          549662955|
+-------------------+

score
+------------+
|count(score)|
+------------+
|   549662934|
+------------+

selftext
+---------------+
|count(selftext)|
+---------------+
|      549662955|
+---------------+

subreddit
+----------------+
|count(subreddit)|
+----------------+
|       549356476|
+----------------+

subreddit_id
+-------------------+
|count(subreddit_id)|
+-------------------+
|          549356476|
+-------------------+

title
+------------+
|count(title)|
+------------+
|   549662955|
+------------+

